# Databricks retail star schema benchmark example

Generate repeatable synthetic tables, write Delta, and run query templates in your workspace. Results are environment-specific.

In [ ]:
# Databricks notebook source
%pip install great-generator

In [ ]:
# Run from the repository root after cloning into your workspace.
import subprocess, sys
subprocess.check_call([sys.executable, "benchmarks/datasets/retail_star_schema/generate_dataset.py", "--output", "/dbfs/tmp/great-generator/retail_star_schema", "--format", "parquet", "--overwrite"])

In [ ]:
base = "dbfs:/tmp/great-generator/retail_star_schema"
for table in ["dim_customer", "dim_product", "dim_store", "dim_date", "fact_sales"]:
    spark.read.parquet(f"{base}/{table}").write.format("delta").mode("overwrite").saveAsTable(f"default.{table}")

In [ ]:
spark.sql('''
SELECT business_date, COUNT(*) AS sales_rows, SUM(sales_amount) AS sales_amount
FROM default.fact_sales
WHERE business_date BETWEEN DATE '2026-01-01' AND DATE '2026-01-07'
GROUP BY business_date
ORDER BY business_date
''').show()